# 03 — Fair Probability and Reference Price

This notebook explains how the public sample uses Binance BTCUSDT-style reference prices as a faster proxy layer for the BTC short-horizon fair probability workflow.

The goal is not to prove a persistent lead-lag alpha. The goal is to make the reference-price assumption inspectable and to show how the fair probability model behaves across time-to-resolution and price-distance regimes.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "sample"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

pd.set_option("display.max_columns", 80)


In [ ]:
from models.fair_probability import estimate_up_probability

ticks = pd.read_csv(DATA_DIR / "tick_snapshots_sample.csv", parse_dates=["timestamp"])
settlements = pd.read_csv(DATA_DIR / "settlements_sample.csv")

print(f"tick rows: {len(ticks):,}")
print(f"markets: {ticks['market_id'].nunique():,}")
ticks.head()


## Reference price fields

The public sample keeps both Polymarket-style quote fields and Binance-style reference fields:

- `bn_price`: Binance BTCUSDT spot tick proxy.
- `bn_open_price`: Binance-derived opening anchor proxy.
- `pm_open_price`: prediction-market opening reference field when available.
- `fair_yes`: model-estimated fair probability for UP / YES.
- `pm_implied_up`: market-implied UP probability from public quote fields.

The private workflow uses Binance as a faster reference proxy, while Polymarket BTC markets settle against an oracle-style reference rather than Binance directly.


In [ ]:
reference_columns = [
    "timestamp",
    "market_id",
    "bn_price",
    "bn_open_price",
    "pm_open_price",
    "current_price",
    "open_anchor_price",
    "remaining_seconds",
    "fair_yes",
    "pm_implied_up",
]
existing_reference_columns = [column for column in reference_columns if column in ticks.columns]
ticks[existing_reference_columns].head(10)


## Price distance and model confidence

The fair probability model uses a standardized distance from the opening anchor. The public sample already exports `z`, `sigma_eff`, and `tau_seconds`, making it possible to inspect when the model becomes very confident.


In [ ]:
summary = ticks[["remaining_seconds", "bn_price", "bn_open_price", "fair_yes", "pm_implied_up", "z"]].describe()
summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_data = ticks.dropna(subset=["remaining_seconds", "fair_yes"])
ax.scatter(plot_data["remaining_seconds"], plot_data["fair_yes"], alpha=0.25, s=12)
ax.set_title("Fair probability by remaining seconds")
ax.set_xlabel("Remaining seconds")
ax.set_ylabel("fair_yes")
ax.grid(True, alpha=0.3)
plt.show()


## Fair probability vs market-implied probability

A useful diagnostic is the gap between model fair probability and market-implied probability. Large gaps are candidate theoretical edges, but they are not executable edge until spread, fill probability, latency, order success, and settlement are considered.


In [ ]:
ticks["fair_minus_market"] = ticks["fair_yes"] - ticks["pm_implied_up"]
edge_view = (
    ticks.groupby("market_id")
    .agg(
        rows=("market_id", "size"),
        avg_fair_yes=("fair_yes", "mean"),
        avg_pm_implied_up=("pm_implied_up", "mean"),
        avg_fair_minus_market=("fair_minus_market", "mean"),
        min_remaining=("remaining_seconds", "min"),
        max_remaining=("remaining_seconds", "max"),
    )
    .sort_values("avg_fair_minus_market", ascending=False)
)
edge_view.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(ticks["fair_minus_market"].dropna(), bins=40)
ax.set_title("Fair probability minus market-implied probability")
ax.set_xlabel("fair_yes - pm_implied_up")
ax.set_ylabel("Rows")
ax.grid(True, alpha=0.3)
plt.show()


## Recomputing one fair probability

The public module can recompute fair probability from public sample fields. This makes the model logic auditable without exposing private execution code.


In [ ]:
sample_row = ticks.dropna(subset=["current_price", "open_anchor_price", "sigma_short", "sigma_long", "remaining_seconds"]).iloc[0]
recomputed = estimate_up_probability(
    current_price=float(sample_row["current_price"]),
    open_anchor_price=float(sample_row["open_anchor_price"]),
    sigma_short=float(sample_row["sigma_short"]),
    sigma_long=float(sample_row["sigma_long"]),
    remaining_seconds=float(sample_row["remaining_seconds"]),
)
comparison = pd.DataFrame(
    [
        {"field": "exported_fair_yes", "value": sample_row["fair_yes"]},
        {"field": "recomputed_fair_yes", "value": recomputed},
        {"field": "absolute_difference", "value": abs(float(sample_row["fair_yes"]) - recomputed)},
    ]
)
comparison


## Author takeaway

The reference-price layer is the starting point of the strategy, but it is not the end of the problem. Binance-style spot movement can create fast theoretical edge signals, yet the project only becomes interesting when those signals are tested against executable conditions. The open research problem is the simulation-to-live gap: whether fast reference-price signals can survive the execution funnel.
